In [1]:
import json
from pathlib import Path
import re
import time

from tqdm.auto import tqdm

from characterization import load_prompt, render_user_prompt, analyze_snapshot
from dsa.utils import load_json

# Inputs

In [2]:
# Inputs
SYSTEM_PROMPT_PATH = "system_message.md"
USER_PROMPT_PATH = "user_message.md"
MODEL_NAME = "gpt-4o-mini"
SLEEP_SECONDS = 0.2  # safety throttle

SNAPSHOTS_DIR = "../../../data/snapshots/refugee_batch1/"
METADATA_DIR = "/home/ajd/corpus/refugee/metadata_indexed_fname/"
SOURCE_LABEL = "Refugee"
OUTPUT_PATH = f"data/results_{SOURCE_LABEL.lower()}_batch1.jsonl"

# Load data

In [3]:
system_prompt = load_prompt(SYSTEM_PROMPT_PATH)
user_prompt_template = load_prompt(USER_PROMPT_PATH)

In [4]:
metadata_files = list(Path(METADATA_DIR).glob("*.json"))
metadata_lookup = {x.stem: x for x in metadata_files}
len(metadata_files)

38

In [5]:
snapshot_files = list(Path(SNAPSHOTS_DIR).glob("*.png"))
len(snapshot_files)

499

In [6]:
## Sampling
# import random
# snapshot_files = random.sample(snapshot_files, 125)
# len(snapshot_files)

# Main pipeline

In [7]:
# Build skip list
to_skip = set()
if Path(OUTPUT_PATH).exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            to_skip.add(json.loads(line)["snapshot_id"])

with open(OUTPUT_PATH, "a", encoding="utf-8") as out_f:
    for s in tqdm(snapshot_files):
        snapshot_id = s.name

        if snapshot_id in to_skip:
            print(f"Skipped: {s.name}")
            continue

        # Base row — every row has the same keys
        row = {
            "snapshot_id": snapshot_id,
            "image_path": str(s),
            "source_label": SOURCE_LABEL,
            "llm_output": None,
            "raw_output": None,
            "usage": None,
            "cost": None,
            "error": None,
        }

        try:
            # Infer metadata file from snapshot filename
            parts = re.split("(_figure|_table)", s.stem)
            fname = parts[0]
            label = parts[1].lstrip("_")
            metadata_path = metadata_lookup.get(fname)
            if metadata_path:
                metadata = load_json(metadata_path)

            # Build prompt variables
            v_title = (
                metadata["document_description"]["title_statement"]["title"]
                or "unknown"
            )
            v_date = (
                metadata.get("document_description").get("date_created")
                or metadata.get("document_description").get("date_available")
                or metadata.get("document_description").get("date_published")
                or "unknown"
            )
            v_language = "unknown"
            if metadata["document_description"]["languages"]:
                v_language = ", ".join(
                    [x["name"] for x in metadata["document_description"]["languages"]]
                )
            v_abstract = metadata["document_description"]["abstract"] or "unknown"
            prompt_vars = {
                "SOURCE_LABEL": SOURCE_LABEL,
                "DOCUMENT_TITLE": v_title,
                "YEAR": v_date,
                "LANGUAGE": v_language,
                "ABSTRACT": v_abstract,
                "SNAPSHOT_TYPE": label,
            }

            # Create user prompt
            user_prompt = render_user_prompt(user_prompt_template, prompt_vars)

            # Responses API
            result = analyze_snapshot(
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                image_path=str(s),
                model=MODEL_NAME,
            )

            row["llm_output"] = result["parsed_output"]
            row["raw_output"] = result["raw_output"]
            row["usage"] = result["usage"].model_dump()
            row["cost"] = result["cost"]
            row["error"] = result["error"]  # None on success, message on parse failure

        except Exception as e:
            row["error"] = str(e)

        # Single write point — always executes
        out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
        out_f.flush()

        time.sleep(SLEEP_SECONDS)


  0%|          | 0/499 [00:00<?, ?it/s]

Skipped: 069_Pakistan-Strengthening-Institutions-for-Refugee-Administration-Project_table_016.png
Skipped: 189_multi-page_table_002.png
Skipped: 203_multi-page_table_004.png
Skipped: 203_multi-page_table_002.png
Skipped: 002_BOSIB-ca473522-8ad0-4c80-9f0d-88bf887f2a2f_table_004.png
Skipped: 200_multi-page_table_001.png
Skipped: 013_BOSIB0efb09b920d90858a0135df22da7d1_table_003.png
Skipped: 051_Pakistan-Balochistan-Human-Capital-Investment-Project_table_003.png
Skipped: 051_Pakistan-Balochistan-Human-Capital-Investment-Project_table_006.png
Skipped: 161_28046_figure_000.png
Skipped: 006_BOSIB-2c22668f-f4ba-42f5-a0d1-7949b7b8fe34_table_006.png
Skipped: 196_multi-page_table_006.png
Skipped: 195_multi-page_figure_000.png
Skipped: 002_BOSIB-ca473522-8ad0-4c80-9f0d-88bf887f2a2f_table_003.png
Skipped: 069_Pakistan-Strengthening-Institutions-for-Refugee-Administration-Project_table_014.png
Skipped: 020_P1781250bdd2b50b0b9720d5c17632331c_table_001.png
Skipped: 202_multi0page_table_004.png
Skippe